# Modification survey: PSU / 5MC / OMG

Inspection of the downloaded PDB entries before rebuilding the h5 dataset.

1. Download inventory: what was requested, what is on disk, in which format
2. Parse every structure (first model) and count PSU, 5MC and OMG per entry and per chain
3. How many survive the current `build_structures_dataset.py` filters
4. Atom completeness of the coarse-grained atoms used by the model, HETATM flags
5. Other modified nucleotides present in RNA chains
6. Counts per gRNAde split and in the current h5

Parsing results are cached in `datasets/mod_survey_*.csv`; delete them to re-parse.

In [ ]:
import os, re, glob
from collections import Counter
from multiprocessing import Pool

import gemmi
import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)

## Configuration

In [ ]:
ROOT = "/scratch/izar/maazoun"
DL_DIR = f"{ROOT}/pdb_download"
RISOTTO = f"{ROOT}/risotto"

PDB_GLOB = f"{DL_DIR}/pdb_files/*.pdb.gz"
CIF_GLOB = f"{DL_DIR}/cif_files/*.cif.gz"
REQUESTED = f"{DL_DIR}/all_pdb.txt"
H5_PATH = f"{RISOTTO}/datasets/pdb_bprna_structures_nuc_v7_psu.h5"
SPLITS = {s: f"{RISOTTO}/datasets/grnade_das_{s}_set.txt" for s in ["train", "validation", "test"]}

CACHE_ENTRIES = f"{RISOTTO}/datasets/mod_survey_entries.csv"
CACHE_CHAINS = f"{RISOTTO}/datasets/mod_survey_chains.csv"
CACHE_RESIDUES = f"{RISOTTO}/datasets/mod_survey_residues.csv"

STD_RNA = ["A", "G", "C", "U"]
MODS = ["PSU", "5MC", "OMG"]

# coarse-grained atoms the model keeps per residue (src/data_encoding.py)
#   PSU: base attached through C5; 5MC is a pyrimidine (N1); OMG is a purine (N9)
CG_ATOMS = {
    "PSU": ["P", "C4'", "O2'", "C5"],
    "5MC": ["P", "C4'", "O2'", "N1"],
    "OMG": ["P", "C4'", "O2'", "N9"],
}

# filters applied by build_structures_dataset.py
MAX_FILE_SIZE = 1e6        # bytes, gzipped file
MAX_NUM_ATOMS = 1024 * 8 * 2

N_WORKERS = min(8, os.cpu_count())

## 1. Download inventory

In [ ]:
requested = sorted({x.strip().upper() for x in open(REQUESTED).read().split(",") if x.strip()})
pdb_paths = {os.path.basename(p).split(".")[0].upper(): p for p in glob.glob(PDB_GLOB)}
cif_paths = {os.path.basename(p).split(".")[0].upper(): p for p in glob.glob(CIF_GLOB)}

both = set(pdb_paths) & set(cif_paths)
on_disk = set(pdb_paths) | set(cif_paths)
missing = sorted(set(requested) - on_disk)
extra = sorted(on_disk - set(requested))

print(f"requested IDs        : {len(requested)}")
print(f"  as .pdb.gz         : {len(pdb_paths)}")
print(f"  as .cif.gz         : {len(cif_paths)}")
print(f"  in both formats    : {len(both)}")
print(f"  missing            : {len(missing)} {missing[:20]}")
print(f"  on disk, not asked : {len(extra)} {extra[:20]}")

# one path per entry, .pdb preferred (that is what the current builder reads)
paths = {**cif_paths, **pdb_paths}
sizes = pd.Series({k: os.path.getsize(p) for k, p in paths.items()})
print(f"\ntotal size on disk   : {sizes.sum() / 1e9:.2f} GB")

## 2. Parse all structures

First model only, as in `build_structures_dataset.py`. For every entry: method, resolution, atom count (including H and water,
as the builder counts before cleaning), chains. For every RNA chain: length and modification counts. For every PSU/5MC/OMG
residue: HETATM flag and which coarse-grained atoms are present. Other non-standard residues inside RNA polymer chains are
recorded too.

In [ ]:
def parse_entry(item):
    pdb_id, path = item
    entry = {"pdb_id": pdb_id, "format": "pdb" if path.endswith(".pdb.gz") else "cif",
             "file_size": os.path.getsize(path), "error": ""}
    chains, residues = [], []
    try:
        st = gemmi.read_structure(path)
        st.setup_entities()
        model = st[0]
        entry.update(
            # .pdb files without REMARK 200/230 only carry the method in EXPDTA
            method=";".join(e.method for e in st.meta.experiments)
                   or (st.info["_exptl.method"] if "_exptl.method" in st.info else ""),
            resolution=st.resolution if st.resolution > 0 else np.nan,
            n_models=len(st),
            n_atoms=model.count_atom_sites(),
            n_chains=len(model),
            title=st.info["_struct.title"] if "_struct.title" in st.info else "",
        )
        for ch in model:
            poly = ch.get_polymer()
            if poly.check_polymer_type() != gemmi.PolymerType.Rna:
                continue
            names = Counter(r.name for r in poly)
            chains.append({"pdb_id": pdb_id, "chain": ch.name, "n_res": len(poly),
                           "n_std": sum(names[n] for n in STD_RNA),
                           **{f"n_{m}": names[m] for m in MODS},
                           "other_mods": ";".join(f"{n}:{c}" for n, c in names.items() if n not in STD_RNA + MODS)})
            for r in poly:
                if r.name in MODS:
                    atoms = {a.name for a in r}
                    residues.append({"pdb_id": pdb_id, "chain": ch.name, "resname": r.name,
                                     "seq_num": r.seqid.num, "icode": r.seqid.icode.strip(),
                                     "het_flag": r.het_flag, "n_atoms": len(r),
                                     "missing_cg": ";".join(a for a in CG_ATOMS[r.name] if a not in atoms)})
    except Exception as e:
        entry["error"] = repr(e)
    return entry, chains, residues


if all(os.path.exists(p) for p in [CACHE_ENTRIES, CACHE_CHAINS, CACHE_RESIDUES]):
    entries = pd.read_csv(CACHE_ENTRIES, keep_default_na=False, na_values=[""])
    chains = pd.read_csv(CACHE_CHAINS, keep_default_na=False, na_values=[""])
    residues = pd.read_csv(CACHE_RESIDUES, keep_default_na=False, na_values=[""])
    print("loaded from cache")
else:
    # big structures first so the pool does not wait on a ribosome at the end
    items = sorted(paths.items(), key=lambda kv: -sizes[kv[0]])
    with Pool(N_WORKERS) as pool:
        out = pool.map(parse_entry, items, chunksize=4)
    entries = pd.DataFrame([o[0] for o in out])
    chains = pd.DataFrame([c for o in out for c in o[1]])
    residues = pd.DataFrame([r for o in out for r in o[2]])
    entries.to_csv(CACHE_ENTRIES, index=False)
    chains.to_csv(CACHE_CHAINS, index=False)
    residues.to_csv(CACHE_RESIDUES, index=False)

# keep ids as strings (e.g. 1E95 would otherwise be read as a float)
for df in (entries, chains, residues):
    df["pdb_id"] = df["pdb_id"].astype(str)
chains["chain"] = chains["chain"].astype(str)
chains["other_mods"] = chains["other_mods"].fillna("")
residues["missing_cg"] = residues["missing_cg"].fillna("")

print(f"entries parsed : {len(entries)}  (errors: {(entries.error.fillna('') != '').sum()})")
print(f"RNA chains     : {len(chains)}")
print(f"mod residues   : {len(residues)}")
entries[entries.error.fillna("") != ""].head()

## 3. Entry overview

In [ ]:
# per-entry modification counts
mod_counts = chains.groupby("pdb_id")[[f"n_{m}" for m in MODS] + ["n_res"]].sum()
entries = entries.drop(columns=[c for c in mod_counts.columns if c in entries], errors="ignore") \
                 .merge(mod_counts, left_on="pdb_id", right_index=True, how="left")
entries[mod_counts.columns] = entries[mod_counts.columns].fillna(0).astype(int)
entries["has_rna"] = entries.n_res > 0
entries["n_mod"] = entries[[f"n_{m}" for m in MODS]].sum(axis=1)

# would the current builder keep this entry? (reads .pdb only, file < 1 MB, atoms < 16384)
entries["builder_keeps"] = (entries.format == "pdb") & (entries.file_size < MAX_FILE_SIZE) & (entries.n_atoms < MAX_NUM_ATOMS)

print(f"entries with an RNA chain: {entries.has_rna.sum()} / {len(entries)}")
print(f"kept by current builder  : {entries.builder_keeps.sum()}\n")
print(entries.method.value_counts().to_string())

In [ ]:
pd.DataFrame({
    "entries": entries.groupby("format").size(),
    "median atoms": entries.groupby("format").n_atoms.median(),
    "median resolution": entries.groupby("format").resolution.median(),
    "kept by builder": entries.groupby("format").builder_keeps.sum(),
})

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for fmt, d in entries.groupby("format"):
    ax[0].hist(np.log10(d.n_atoms.clip(lower=1)), bins=60, alpha=0.6, label=f".{fmt} ({len(d)})")
ax[0].axvline(np.log10(MAX_NUM_ATOMS), color="k", ls="--", label=f"builder limit {MAX_NUM_ATOMS}")
ax[0].set_xlabel("log10(atoms in first model)"); ax[0].set_ylabel("entries"); ax[0].legend()

ax[1].hist(entries.resolution.dropna(), bins=60, range=(0, 10))
ax[1].set_xlabel("resolution (Å)"); ax[1].set_ylabel("entries")
ax[1].set_title(f"{entries.resolution.isna().sum()} entries without resolution (NMR, ...)")
plt.tight_layout()

## 4. Modification counts

In [ ]:
def mod_table(e, c):
    rows = {}
    for m in MODS:
        col = f"n_{m}"
        rows[m] = {"residues": int(c[col].sum()), "chains": int((c[col] > 0).sum()), "entries": int((e[col] > 0).sum())}
    rows["any"] = {"residues": int(c[[f"n_{m}" for m in MODS]].sum().sum()),
                   "chains": int((c[[f"n_{m}" for m in MODS]].sum(axis=1) > 0).sum()),
                   "entries": int((e.n_mod > 0).sum())}
    return pd.DataFrame(rows).T

kept_ids = set(entries.pdb_id[entries.builder_keeps])
tables = {
    "all downloaded": mod_table(entries, chains),
    ".pdb files": mod_table(entries[entries.format == "pdb"], chains[chains.pdb_id.isin(entries.pdb_id[entries.format == "pdb"])]),
    ".cif files": mod_table(entries[entries.format == "cif"], chains[chains.pdb_id.isin(entries.pdb_id[entries.format == "cif"])]),
    "kept by builder": mod_table(entries[entries.builder_keeps], chains[chains.pdb_id.isin(kept_ids)]),
}
pd.concat(tables, axis=1)

In [ ]:
# fraction of RNA residues that are modified (in chains of the given set)
for name, ids in [("all", set(entries.pdb_id)), ("kept by builder", kept_ids)]:
    c = chains[chains.pdb_id.isin(ids)]
    tot = c.n_res.sum()
    print(f"{name:<16} RNA residues={tot:8d}  " + "  ".join(f"{m}={c[f'n_{m}'].sum() / tot:.4%}" for m in MODS))

In [ ]:
# co-occurrence: which combinations of modifications appear in the same entry
combo = entries[entries.n_mod > 0].apply(lambda r: "+".join(m for m in MODS if r[f"n_{m}"] > 0), axis=1)
combo.value_counts().rename("entries").to_frame()

In [ ]:
fig, ax = plt.subplots(1, len(MODS), figsize=(14, 3.5))
for a, m in zip(ax, MODS):
    v = chains[f"n_{m}"][chains[f"n_{m}"] > 0]
    a.hist(v, bins=np.arange(1, v.max() + 2) - 0.5 if len(v) else 10)
    a.set_title(f"{m}: {len(v)} chains"); a.set_xlabel(f"{m} per chain"); a.set_ylabel("chains")
plt.tight_layout()

In [ ]:
# entries with the most modifications
cols = ["pdb_id", "format", "method", "resolution", "n_atoms", "n_res"] + [f"n_{m}" for m in MODS] + ["builder_keeps", "title"]
entries.sort_values("n_mod", ascending=False)[cols].head(25)

In [ ]:
# entries the builder would keep that carry modifications
entries[entries.builder_keeps & (entries.n_mod > 0)].sort_values("n_mod", ascending=False)[cols].head(40)

## 5. Residue-level checks

* `het_flag`: modified residues written as HETATM get split off into their own chain by `tag_hetatm_chains`,
  unless excluded there (currently only PSU is).
* `missing_cg`: residues lacking one of the coarse-grained atoms are dropped by `extract_backbone_cg`.

In [ ]:
pd.crosstab(residues.resname, residues.het_flag, margins=True)

In [ ]:
residues["cg_complete"] = residues.missing_cg == ""
summary = residues.groupby("resname").agg(residues=("pdb_id", "size"), cg_complete=("cg_complete", "sum"))
summary["cg_complete_frac"] = summary.cg_complete / summary.residues
print(summary.to_string(), "\n")
print("most common missing coarse-grained atoms:")
print(residues[~residues.cg_complete].groupby(["resname", "missing_cg"]).size().sort_values(ascending=False).head(15).to_string())

In [ ]:
# atoms per modified residue (heavy atoms + any H/D present in the file)
residues.groupby("resname").n_atoms.describe()

## 6. Other modified nucleotides in RNA chains

In [ ]:
other = Counter()
other_entries = Counter()
for pdb_id, s in zip(chains.pdb_id, chains.other_mods):
    seen = set()
    for item in filter(None, s.split(";")):
        name, n = item.rsplit(":", 1)
        other[name] += int(n)
        seen.add(name)
    other_entries.update(seen)

other_df = pd.DataFrame({"residues": pd.Series(other), "entries": pd.Series(other_entries)}).sort_values("residues", ascending=False)
mods_df = pd.DataFrame({m: {"residues": tables["all downloaded"].loc[m, "residues"],
                            "entries": tables["all downloaded"].loc[m, "entries"]} for m in MODS}).T
print("tracked:"); print(mods_df.to_string()); print()
print(f"{len(other_df)} other residue names inside RNA polymer chains; top 30:")
other_df.head(30)

## 7. Splits and current h5

In [ ]:
split_ids = {s: np.unique([x.split("_")[0].upper() for x in np.genfromtxt(p, dtype="U")]) for s, p in SPLITS.items()}

rows = {}
for s, ids in split_ids.items():
    e = entries[entries.pdb_id.isin(ids)]
    rows[s] = {"split PDB ids": len(ids), "downloaded": len(e), "kept by builder": int(e.builder_keeps.sum()),
               **{f"{m} res": int(e[f"n_{m}"].sum()) for m in MODS},
               **{f"{m} res (kept)": int(e[f"n_{m}"][e.builder_keeps].sum()) for m in MODS},
               **{f"{m} entries (kept)": int(((e[f"n_{m}"] > 0) & e.builder_keeps).sum()) for m in MODS}}
in_any = np.unique(np.concatenate(list(split_ids.values())))
e = entries[~entries.pdb_id.isin(in_any)]
rows["not in any split"] = {"split PDB ids": 0, "downloaded": len(e), "kept by builder": int(e.builder_keeps.sum()),
                            **{f"{m} res": int(e[f"n_{m}"].sum()) for m in MODS},
                            **{f"{m} res (kept)": int(e[f"n_{m}"][e.builder_keeps].sum()) for m in MODS},
                            **{f"{m} entries (kept)": int(((e[f"n_{m}"] > 0) & e.builder_keeps).sum()) for m in MODS}}
pd.DataFrame(rows).T

In [ ]:
# what is in the current h5 (built with PSU only)
with h5py.File(H5_PATH, "r") as hf:
    h5_keys = np.array(hf["metadata/keys"]).astype("U")
    h5_resnames = np.array(hf["metadata/std_resnames"]).astype("U").tolist()
h5_ids = set(k.split("/")[1] for k in h5_keys)

print(f"h5 entries                  : {len(h5_ids)}")
print(f"h5 std_resnames             : {h5_resnames}")
print(f"kept by builder, not in h5  : {len(kept_ids - h5_ids)}  (skipped during build: read errors, no valid subunits, ...)")
print(f"in h5, not 'kept by builder': {len(h5_ids - kept_ids)}")
e = entries[entries.pdb_id.isin(h5_ids)]
print("modifications in h5 entries : " + ", ".join(f"{m}={e[f'n_{m}'].sum()} res / {(e[f'n_{m}'] > 0).sum()} entries" for m in MODS))

## 8. Summary

In [ ]:
t = tables["all downloaded"]; k = tables["kept by builder"]
print(f"requested {len(requested)} | on disk {len(on_disk)} ({len(pdb_paths)} .pdb, {len(cif_paths)} .cif) | missing {len(missing)}")
print(f"entries with RNA: {entries.has_rna.sum()} | kept by current builder: {entries.builder_keeps.sum()}\n")
print(f"{'':6}{'all residues':>14}{'all entries':>13}{'kept residues':>15}{'kept entries':>14}")
for m in MODS:
    print(f"{m:6}{t.loc[m, 'residues']:>14}{t.loc[m, 'entries']:>13}{k.loc[m, 'residues']:>15}{k.loc[m, 'entries']:>14}")